# ARIA v3.0 — Week 5 Notebook
## Dynamic Disaster Impact Monitoring for Typhoon Fung-wong

This notebook builds **ARIA v3.0**, a dynamic risk monitoring system that integrates:

1. Week 3 river-distance shelter risk  
2. Week 4 terrain risk  
3. Live or simulated rainfall station data  
4. Spatial overlay and dynamic shelter risk classification  
5. Folium interactive monitoring map

**Output:** `ARIA_v3_Fungwong.html`


## Captain's Log 01 — Mission Setup

This cell loads all required libraries, environment variables, and basic settings.  
All analysis settings are controlled through `.env` so the notebook can switch between **LIVE** and **SIMULATION** modes safely.


In [ ]:
# Cell 1: Imports and environment setup

import os
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import folium
from folium.plugins import HeatMap
import requests

warnings.filterwarnings("ignore")

try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError("Please install python-dotenv first: pip install python-dotenv")

load_dotenv()

APP_MODE = os.getenv("APP_MODE", "SIMULATION").upper()
CWA_API_KEY = os.getenv("CWA_API_KEY", "")
SIMULATION_DATA = os.getenv("SIMULATION_DATA", "fungwong_202511.json")
SHELTER_FILE = os.getenv("SHELTER_FILE", "shelters_composite_risk.csv")
BUFFER_METERS = int(os.getenv("BUFFER_METERS", "5000"))
RAIN_URGENT = float(os.getenv("RAIN_URGENT", "40"))
RAIN_CRITICAL = float(os.getenv("RAIN_CRITICAL", "80"))
TARGET_CENTER_LAT = float(os.getenv("TARGET_CENTER_LAT", "23.987"))
TARGET_CENTER_LON = float(os.getenv("TARGET_CENTER_LON", "121.601"))
OUTPUT_HTML = os.getenv("OUTPUT_HTML", "ARIA_v3_Fungwong.html")

print("APP_MODE =", APP_MODE)
print("SIMULATION_DATA =", SIMULATION_DATA)
print("SHELTER_FILE =", SHELTER_FILE)
print("BUFFER_METERS =", BUFFER_METERS)
print("RAIN_URGENT =", RAIN_URGENT)
print("RAIN_CRITICAL =", RAIN_CRITICAL)


## Captain's Log 02 — Load Shelter Risk Baseline

This cell loads the shelter dataset produced from earlier weeks.  
It should already contain the shelter coordinates and baseline risk attributes such as river and terrain risk.


In [ ]:
# Cell 2: Load shelter baseline data

shelter_path = Path(SHELTER_FILE)
if not shelter_path.exists():
    # fallback for uploaded file in notebook folder
    shelter_path = Path("/mnt/data") / SHELTER_FILE

if not shelter_path.exists():
    raise FileNotFoundError(f"Cannot find shelter file: {SHELTER_FILE}")

shelters_df = pd.read_csv(shelter_path, encoding="utf-8-sig")
print("Shelter rows:", len(shelters_df))
print("Columns:")
print(list(shelters_df.columns))
shelters_df.head()


## Captain's Log 03 — Standardize Shelter Fields

Different previous-week outputs may use slightly different column names.  
This cell standardizes the geometry and risk columns so later analysis can run without rewriting the logic.


In [ ]:
# Cell 3: Standardize shelter fields and create GeoDataFrame

def find_first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

lon_col = find_first_existing(shelters_df, ["lon", "longitude", "x", "lng", "LON", "Longitude"])
lat_col = find_first_existing(shelters_df, ["lat", "latitude", "y", "LAT", "Latitude"])

if lon_col is None or lat_col is None:
    raise ValueError("Shelter CSV must contain latitude/longitude columns.")

terrain_col = find_first_existing(shelters_df, ["terrain_risk", "risk_level", "terrain_level"])
name_col = find_first_existing(shelters_df, ["name", "shelter_name", "避難所名稱"])
id_col = find_first_existing(shelters_df, ["shelter_id", "id", "ID"])

shelters = gpd.GeoDataFrame(
    shelters_df.copy(),
    geometry=gpd.points_from_xy(shelters_df[lon_col], shelters_df[lat_col]),
    crs="EPSG:4326"
)

# Normalize terrain risk to English categories used in Week 5 logic
terrain_map = {
    "極高風險": "HIGH",
    "高風險": "HIGH",
    "中風險": "MEDIUM",
    "低風險": "LOW",
    "HIGH": "HIGH",
    "MEDIUM": "MEDIUM",
    "LOW": "LOW"
}

if terrain_col is None:
    shelters["terrain_risk"] = "UNKNOWN"
else:
    shelters["terrain_risk"] = shelters[terrain_col].astype(str).str.strip().map(terrain_map).fillna(
        shelters[terrain_col].astype(str).str.upper()
    )

if name_col is None:
    shelters["display_name"] = "Unknown Shelter"
else:
    shelters["display_name"] = shelters[name_col].astype(str)

if id_col is None:
    shelters["display_id"] = shelters.index.astype(str)
else:
    shelters["display_id"] = shelters[id_col].astype(str)

print("Shelter CRS:", shelters.crs)
print(shelters[["display_id", "display_name", "terrain_risk"]].head())


## Captain's Log 04 — Build the Mode Switcher

This cell creates the data loader for **LIVE** and **SIMULATION** mode.  
It also includes a fallback mechanism so the notebook can still run when the CWA API fails.


In [ ]:
# Cell 4: Load rainfall JSON from LIVE or SIMULATION mode

CWA_URL = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"

def load_rainfall_json():
    sim_path = Path(SIMULATION_DATA)
    if not sim_path.exists():
        sim_path = Path("/mnt/data") / SIMULATION_DATA

    if APP_MODE == "LIVE":
        try:
            params = {"Authorization": CWA_API_KEY, "format": "JSON"}
            r = requests.get(CWA_URL, params=params, timeout=20)
            r.raise_for_status()
            print("Loaded LIVE rainfall data from CWA API.")
            return r.json(), "LIVE"
        except Exception as e:
            print("LIVE API failed, switching to fallback simulation snapshot.")
            print("Reason:", e)

            if sim_path.exists():
                with open(sim_path, "r", encoding="utf-8") as f:
                    return json.load(f), "FALLBACK_SIMULATION"
            else:
                raise FileNotFoundError(
                    "LIVE API failed and fallback simulation file is missing."
                )

    else:
        if not sim_path.exists():
            raise FileNotFoundError(
                f"SIMULATION mode selected but file not found: {SIMULATION_DATA}"
            )
        with open(sim_path, "r", encoding="utf-8") as f:
            print("Loaded SIMULATION rainfall data.")
            return json.load(f), "SIMULATION"

rain_json, rain_source = load_rainfall_json()
print("Rain source used:", rain_source)
print("Top-level keys:", list(rain_json.keys())[:10])


## Captain's Log 05 — Normalize CWA and CoLife JSON

The live API and simulation snapshot share a similar structure but differ in:
- coordinate sets
- numeric types
- missing-value handling

This function converts both into one consistent station table for all later analysis.


In [ ]:
# Cell 5: Normalize rainfall station JSON

def safe_float(x):
    try:
        return float(x)
    except:
        return np.nan

def normalize_cwa_json(data):
    stations = data.get("records", {}).get("Station", [])
    rows = []

    for st in stations:
        station_name = st.get("StationName", "Unknown")
        station_id = st.get("StationId", "Unknown")

        # Coordinates
        lat = None
        lon = None

        coords = st.get("GeoInfo", {}).get("Coordinates", [])
        if isinstance(coords, list) and len(coords) >= 2:
            # CWA LIVE often has [0]=TWD67, [1]=WGS84
            cand = coords[1]
            lat = safe_float(cand.get("StationLatitude"))
            lon = safe_float(cand.get("StationLongitude"))
        elif isinstance(coords, list) and len(coords) == 1:
            cand = coords[0]
            lat = safe_float(cand.get("StationLatitude"))
            lon = safe_float(cand.get("StationLongitude"))

        # Some payloads may store coordinates elsewhere
        if pd.isna(lat) or pd.isna(lon):
            lat = safe_float(st.get("lat", np.nan))
            lon = safe_float(st.get("lon", np.nan))

        rainfall = st.get("RainfallElement", {})
        rain_1hr = safe_float(rainfall.get("Past1hr", np.nan))
        rain_24hr = safe_float(rainfall.get("Past24hr", np.nan))
        rain_now = safe_float(rainfall.get("Now", np.nan))

        rows.append({
            "station_id": station_id,
            "station_name": station_name,
            "lat": lat,
            "lon": lon,
            "rain_1hr": rain_1hr,
            "rain_24hr": rain_24hr,
            "rain_now": rain_now
        })

    df = pd.DataFrame(rows)

    # Filter invalid rows
    df = df.replace(-998, np.nan)
    df = df.dropna(subset=["lat", "lon", "rain_1hr"]).copy()

    return df

rain_df = normalize_cwa_json(rain_json)
print("Valid rainfall stations:", len(rain_df))
rain_df.head()


## Captain's Log 06 — Build Rainfall GeoDataFrame

This cell converts the normalized rainfall table into a GeoDataFrame, reprojects it to EPSG:3826 for spatial analysis, and prepares rain buffers.


In [ ]:
# Cell 6: Build rainfall GeoDataFrame and buffers

rain_gdf = gpd.GeoDataFrame(
    rain_df.copy(),
    geometry=gpd.points_from_xy(rain_df["lon"], rain_df["lat"]),
    crs="EPSG:4326"
)

rain_gdf_3826 = rain_gdf.to_crs("EPSG:3826")
shelters_3826 = shelters.to_crs("EPSG:3826")

assert str(rain_gdf_3826.crs) == str(shelters_3826.crs), "CRS MISMATCH!"

# Use stations above urgent threshold for impact buffer
impact_stations = rain_gdf_3826[rain_gdf_3826["rain_1hr"] > RAIN_URGENT].copy()
impact_stations["geometry"] = impact_stations.geometry.buffer(BUFFER_METERS)

print("Stations above urgent threshold:", len(impact_stations))
impact_stations.head()


## Captain's Log 07 — Spatial Overlay and Affected Shelters

This cell finds which shelters fall inside heavy-rain impact zones using `gpd.sjoin()`.  
It also keeps the associated station and rainfall information for later popup display.


In [ ]:
# Cell 7: Spatial join shelters with rainfall impact buffers

if len(impact_stations) == 0:
    shelters_3826["rain_1hr"] = np.nan
    shelters_3826["station_name"] = None
    shelters_3826["affected_by_rain"] = False
    joined = shelters_3826.copy()
else:
    joined = gpd.sjoin(
        shelters_3826,
        impact_stations[["station_id", "station_name", "rain_1hr", "geometry"]],
        how="left",
        predicate="intersects"
    )

    joined["affected_by_rain"] = ~joined["station_name"].isna()

print("Affected shelters:", joined["affected_by_rain"].sum())
joined.head()


## Captain's Log 08 — Dynamic Risk Classification

This cell applies the Week 5 dynamic risk logic:

- **CRITICAL**: rain_1hr > 80mm and inside impact zone  
- **URGENT**: rain_1hr > 40mm and terrain_risk == HIGH  
- **WARNING**: rain_1hr > 40mm or terrain_risk == HIGH  
- **SAFE**: everything else


In [ ]:
# Cell 8: Dynamic risk classification

def classify_dynamic_risk(row):
    rain = row.get("rain_1hr", np.nan)
    terrain = str(row.get("terrain_risk", "UNKNOWN")).upper()
    affected = bool(row.get("affected_by_rain", False))

    if affected and pd.notna(rain) and rain > RAIN_CRITICAL:
        return "CRITICAL"
    elif pd.notna(rain) and rain > RAIN_URGENT and terrain == "HIGH":
        return "URGENT"
    elif (pd.notna(rain) and rain > RAIN_URGENT) or terrain == "HIGH":
        return "WARNING"
    else:
        return "SAFE"

joined["dynamic_risk"] = joined.apply(classify_dynamic_risk, axis=1)

risk_summary = joined["dynamic_risk"].value_counts(dropna=False)
print(risk_summary)


## Captain's Log 09 — Nearest Rain Station for Better Popups

Some shelters may not fall inside a rain buffer, but the commander still needs context.  
This cell calculates the nearest rainfall station and attaches its name and hourly rainfall to each shelter.


In [ ]:
# Cell 9: Attach nearest rainfall station

rain_pts = rain_gdf_3826[["station_id", "station_name", "rain_1hr", "geometry"]].copy()

if len(rain_pts) > 0:
    nearest = gpd.sjoin_nearest(
        joined.drop(columns=["index_right"], errors="ignore"),
        rain_pts,
        how="left",
        distance_col="dist_to_station_m"
    )
else:
    nearest = joined.copy()
    nearest["station_name_right"] = None
    nearest["rain_1hr_right"] = np.nan
    nearest["dist_to_station_m"] = np.nan

# Prefer joined buffer-matched station info first, fallback to nearest
nearest["popup_station_name"] = nearest["station_name"].fillna(nearest.get("station_name_right"))
nearest["popup_rain_1hr"] = nearest["rain_1hr"].fillna(nearest.get("rain_1hr_right"))

nearest = nearest.to_crs("EPSG:4326")
nearest.head()


## Captain's Log 10 — Build the Interactive Folium Map

This cell creates the final monitoring dashboard with:
- rainfall CircleMarkers
- rainfall HeatMap
- shelter markers colored by dynamic risk
- popup information
- layer control

The final map is saved as `ARIA_v3_Fungwong.html`.


In [ ]:
# Cell 10: Build Folium interactive map

def rain_color(mm):
    if pd.isna(mm):
        return "gray"
    elif mm <= 10:
        return "green"
    elif mm <= 40:
        return "yellow"
    elif mm <= 80:
        return "orange"
    else:
        return "red"

def shelter_color(level):
    return {
        "SAFE": "green",
        "WARNING": "orange",
        "URGENT": "red",
        "CRITICAL": "darkred"
    }.get(level, "gray")

m = folium.Map(location=[TARGET_CENTER_LAT, TARGET_CENTER_LON], zoom_start=9, tiles="CartoDB positron")

# Rainfall layer
rain_fg = folium.FeatureGroup(name="Rainfall Stations", show=True)
for _, row in rain_gdf.iterrows():
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=max(3, min(18, row["rain_1hr"] / 6 if pd.notna(row["rain_1hr"]) else 3)),
        color=rain_color(row["rain_1hr"]),
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>Station:</b> {row['station_name']}<br>"
            f"<b>1-hr Rain:</b> {row['rain_1hr']:.1f} mm<br>"
            f"<b>24-hr Rain:</b> {row['rain_24hr']:.1f} mm" if pd.notna(row["rain_24hr"]) else
            f"<b>Station:</b> {row['station_name']}<br><b>1-hr Rain:</b> {row['rain_1hr']:.1f} mm",
            max_width=300
        )
    ).add_to(rain_fg)
rain_fg.add_to(m)

# Heatmap layer
heat_fg = folium.FeatureGroup(name="Rainfall HeatMap", show=False)
heat_data = rain_df[["lat", "lon", "rain_1hr"]].dropna().values.tolist()
if len(heat_data) > 0:
    HeatMap(heat_data, radius=20, blur=15, max_zoom=10).add_to(heat_fg)
heat_fg.add_to(m)

# Shelter layer
shelter_fg = folium.FeatureGroup(name="Shelters Dynamic Risk", show=True)
for _, row in nearest.iterrows():
    popup_html = (
        f"<b>Shelter:</b> {row['display_name']}<br>"
        f"<b>Terrain Risk:</b> {row['terrain_risk']}<br>"
        f"<b>Dynamic Risk:</b> {row['dynamic_risk']}<br>"
        f"<b>Nearest Rain Station:</b> {row['popup_station_name']}<br>"
        f"<b>1-hr Rain:</b> {row['popup_rain_1hr']:.1f} mm"
        if pd.notna(row['popup_rain_1hr']) else
        f"<b>Shelter:</b> {row['display_name']}<br>"
        f"<b>Terrain Risk:</b> {row['terrain_risk']}<br>"
        f"<b>Dynamic Risk:</b> {row['dynamic_risk']}"
    )

    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(popup_html, max_width=320),
        icon=folium.Icon(color=shelter_color(row["dynamic_risk"]), icon="home", prefix="fa")
    ).add_to(shelter_fg)
shelter_fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m.save(OUTPUT_HTML)
print(f"Saved map to: {OUTPUT_HTML}")
m


## Captain's Log 11 — Risk Summary for Report Writing

This final cell prints a clean summary table that can be used in your README or assignment report.


In [ ]:
# Cell 11: Final summary table

summary = (
    nearest.groupby("dynamic_risk")
    .agg(
        shelters=("display_id", "count"),
        avg_nearest_rain_1hr=("popup_rain_1hr", "mean")
    )
    .reset_index()
)

summary["avg_nearest_rain_1hr"] = summary["avg_nearest_rain_1hr"].round(2)
summary


## Captain's Log 12 — AI Diagnostic Log Notes

You can copy the following issues into `README.md`:

1. **Folium latitude/longitude order issue**  
   Fixed by always passing `[lat, lon]`, not `[lon, lat]`.

2. **CWA missing value `-998` issue**  
   Fixed by replacing `-998` with `NaN` before creating the GeoDataFrame.

3. **Empty `sjoin()` result caused by CRS mismatch**  
   Fixed by reprojecting both shelters and rainfall stations to `EPSG:3826` before buffering and spatial join.

4. **HeatMap blind spots in mountains**  
   Explained as a station distribution problem rather than a mapping bug.
